# Chapter 24 — Strain Gage Characteristics
### *Modern Strain Gage Technology* — Companion Notebook

A strain gage's datasheet is a column of numbers, and two of them — how much power it dissipates and how much it drifts with temperature — quietly decide whether a measurement can be trusted.

Companion notebook to Chapter 24 — Strain Gage Characteristics.

Selecting a strain gage involves more than choosing a resistance and gage factor. The excitation voltage, gage resistance, and thermal conductivity of the specimen material together determine how much power is dissipated in the gage itself — and excessive self-heating raises the gage temperature above the specimen temperature, introducing a thermal gradient that produces apparent strain and, in extreme cases, damages the bond or the gage alloy.

The rule is straightforward: gage power dissipation must stay within limits set by the specimen's ability to conduct heat away. For metals with high thermal conductivity, up to 10 mW is generally acceptable. For composites and plastics, the limit drops to 1 mW or less. For foam and elastomers, it may be as low as 0.1 mW. The table below maps excitation voltage and gage resistance to dissipated power in a quarter-bridge circuit (where the gage dissipates $V_{ex}^2 / 4R_g$), flagging conditions that require reduced excitation or a higher-resistance gage.

The second half turns to *thermal output* — the apparent strain a self-temperature-compensated gage reports when the temperature changes but no load is applied — and reproduces Figure 24.1 of the chapter.

**This notebook demonstrates:**
1. **Self-heating power** vs. excitation voltage and gage resistance — with an OK / CAUTION / HIGH classification.
2. **Thermal output (apparent strain)** of self-temperature-compensated Constantan and Karma gages (Figure 24.1).

In [ ]:
# !pip install numpy matplotlib
import os

import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11, 'axes.titlesize': 12})

---
## 1 — Self-Heating Power Check

Power dissipated in the gage is P = V²_ex / (4R_g), where the factor of 4 accounts for the quarter-bridge voltage divider — only one quarter of the excitation voltage appears across the gage. Higher excitation improves the bridge output signal but increases self-heating proportionally with V². Higher gage resistance reduces heating for the same excitation but also reduces the output signal, so the tradeoff is always between sensitivity and thermal loading.

The table classifies combinations as OK (≤10 mW), CAUTION (10–50 mW), or HIGH (>50 mW). In practice, the OK threshold applies to metals with good thermal contact between gage and specimen. For composites, the threshold is ten times lower. When a test requires simultaneous high sensitivity and low self-heating — transducer design, for example — the solution is usually a higher-resistance gage (1000 Ω or greater) at a modest excitation voltage (2–3 V).

In [ ]:
# Self-heating limits (mW) and the conditions we sweep.
QUARTER_BRIDGE_DIVISOR = 4          # gage sees V_ex/2, so P = V_ex^2 / (4 R_g)
OK_LIMIT_MW = 10.0                  # acceptable on metals with good thermal contact
CAUTION_LIMIT_MW = 50.0            # borderline; reduce V_ex or raise R_g above this

EXCITATION_VOLTAGES_V = [1, 2, 5, 10]
GAGE_RESISTANCES_OHM = [120, 350, 1000]


def gage_power_mW(V_ex, R_g):
    """Power dissipated in a single quarter-bridge gage, in milliwatts.

    Only V_ex/2 appears across the active gage in a quarter bridge, so the
    dissipation is V_ex**2 / (4 * R_g). Inputs are volts and ohms.
    """
    return V_ex**2 / (QUARTER_BRIDGE_DIVISOR * R_g) * 1000.0


def heating_status(P_mW):
    """Classify gage self-heating as OK, CAUTION, or HIGH against the metal limits."""
    if P_mW <= OK_LIMIT_MW:
        return "OK"
    if P_mW <= CAUTION_LIMIT_MW:
        return "CAUTION"
    return "HIGH"


print("SELF-HEATING POWER CHECK")
print(f"{'Vex (V)':>8}  {'Rg (Ω)':>8}  {'P_gage (mW)':>14}  {'Status':>10}")
for V_ex in EXCITATION_VOLTAGES_V:
    for R_g in GAGE_RESISTANCES_OHM:
        P_mW = gage_power_mW(V_ex, R_g)
        print(f"{V_ex:>8}  {R_g:>8}  {P_mW:>14.2f}  {heating_status(P_mW):>10}")
    print()
print("Rule: ≤10 mW on metals, ≤1 mW on composites/plastics, ≤0.1 mW on foam/rubber")

---
## 2 — Figure 24.1: Thermal Output of STC Gages

Thermal output (apparent strain) is the change in gage resistance caused by temperature alone — not by mechanical strain in the specimen. In a self-temperature-compensated (STC) gage, the alloy is cold-worked so that its temperature coefficient of resistance partially cancels the differential thermal expansion between the gage and its substrate. The compensation is exact only at the reference temperature (25 °C); away from that point, a residual error grows in both directions.

Both alloy families produce **U-shaped curves** when the gage is matched to its substrate: minimum near the balance temperature, rising at both cold and hot extremes. The cold-side slope is gentler; the hot side rises more steeply due to the nonlinear TCR of the alloy. Karma (Ni-Cr) is inherently flatter than Constantan (Cu-Ni) because its TCR varies less with temperature, so the STC balance holds over a wider range.

In [ ]:
# ── Temperature axis (°F) ─────────────────────────────────────────────────────
T = np.linspace(-58, 392, 500)      # −50 °C to 200 °C expressed in °F
T_REF_F = 77.0                       # 25 °C balance temperature in °F


def thermal_output(T, T_min, A, T_ref=T_REF_F):
    """Apparent strain (µε) of an STC gage, referenced to zero at T_ref.

    A simple symmetric-parabola model of thermal output: minimum near the
    balance temperature T_min, rising at both cold and hot extremes. The
    curve is offset so it reads zero at the reference temperature T_ref.
    A is the curvature coefficient in µε/°F**2.
    """
    return A * ((T - T_min)**2 - (T_ref - T_min)**2)


# ── Thermal output models ─────────────────────────────────────────────────────
# Same quadratic model for both alloys; coefficients scaled by (5/9)² for the
# °F axis, i.e. A_F = A_C × (5/9)².

# Constantan (Advance) STC-06 on steel — moderate compensation, asymmetric rise
#   minimum at ~41 °F (5 °C); A_F = 0.005 × (5/9)² ≈ 0.001543 µε/°F²
T_MIN_CON_F = 41.0
A_CON = 0.001543
eps_constantan = thermal_output(T, T_MIN_CON_F, A_CON)

# Karma (Ni-Cr) STC-13 on aluminum — superior TCR linearity, flatter residual
#   minimum at ~59 °F (15 °C); A_F = 0.0018 × (5/9)² ≈ 0.000556 µε/°F²
T_MIN_KAR_F = 59.0
A_KAR = 0.000556
eps_karma = thermal_output(T, T_MIN_KAR_F, A_KAR)

# ── Figure ────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 4.5))

ax.axhline(0, color='0.75', linewidth=0.8, zorder=1)
ax.axvline(T_REF_F, color='0.5', linewidth=0.8, linestyle=':', zorder=1,
           label=f'Reference temperature ({T_REF_F:.0f} °F)')

ax.plot(T, eps_constantan, color='#1a3a6b', linewidth=2.0,
        label='Constantan (STC-06, steel substrate)')
ax.plot(T, eps_karma, color='#8b1a1a', linewidth=2.0, linestyle='--',
        label='Karma (STC-13, aluminum substrate)')

ax.set_xlabel('Temperature (°F)', fontsize=11)
ax.set_ylabel('Apparent strain (µε)', fontsize=11)
ax.set_title('Thermal Output (Apparent Strain) of STC Gages', style='italic', fontsize=12)
ax.set_xlim(-58, 392)
ax.set_ylim(-20, 210)
ax.legend(fontsize=9.5, framealpha=0.9)
ax.grid(True, color='0.88', linewidth=0.6)

fig.tight_layout()

out_path = os.path.join('..', 'src', 'media', 'ch24', 'fig24_1_thermal_output.png')
os.makedirs(os.path.dirname(out_path), exist_ok=True)
fig.savefig(out_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved → {out_path}")

---
## Where to take this next

The two calculations above are deliberately small so you can extend them into a working selection tool:

1. **Add the composite and elastomer limits to the power check.** Right now the table classifies against the 10 mW metal limit only. Pass the applicable limit (10 / 1 / 0.1 mW) into `heating_status` as an argument and reprint the table for a carbon-fiber or foam specimen — you will see why high-resistance gages and low excitation become mandatory off metal.

2. **Fit a real thermal-output polynomial.** The curves here are a simple symmetric parabola. Replace `thermal_output` with the third- or fourth-order polynomial a manufacturer prints for a specific STC code (e.g., Micro-Measurements TN-504), evaluate the residual apparent strain over your service temperature range, and convert it to an equivalent stress error with the gage factor and Young's modulus from Chapter 24.

3. **Turn the gage-factor temperature coefficient into a gain-error curve.** Using the $+0.9\%$ per 100 °C figure from Table 24.1, plot the sensitivity error versus temperature and overlay it on the thermal-output plot — a direct picture of the chapter's central distinction between a *gain* error (slope) and a *zero* offset (thermal output).